# Dataset Inspection

This notebook performs automated dataset checks to ensure its up to quality before being used to train the model. Run it from the repository root after downloading the dataset and making sure `data/sid_priority_splits.csv` is available.

Automated checks cover image extensions, dimensions, formats, corruption, split/class counts, missing CSV paths, exact duplicates, and duplicate leakage across splits.

## 1. Configuration

In [1]:
from pathlib import Path
from collections import Counter, defaultdict
import hashlib
import csv
import random

from PIL import Image
import matplotlib.pyplot as plt

# Find the repository root.
REPO_ROOT = Path.cwd()

if not (REPO_ROOT / "data").is_dir():
    REPO_ROOT = Path("/content/ai-image-detector")

# Both physical dataset locations.
DATASET_ROOTS = {
    "CIFAKE": REPO_ROOT / "data" / "raw" / "CIFAKE",
    "SID_Set": REPO_ROOT / "data" / "raw" / "SID_Set_subset",
}

# Use the SID-priority manifest.
SPLITS_FILE = (
    REPO_ROOT
    / "data"
    / "sid_priority_splits.csv"
)

VALID_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp",
}

SEED = 42
random.seed(SEED)

print("Repository root:", REPO_ROOT)
print("Current directory:", Path.cwd())
print("Splits file exists:", SPLITS_FILE.exists())

for dataset_name, dataset_root in DATASET_ROOTS.items():
    print(
        f"{dataset_name} folder exists:",
        dataset_root.exists(),
        dataset_root,
    )

Repository root: /content/ai-image-detector
Current directory: /Users/chengruiyan/Projects/AI-image-detector/notebooks
Splits file exists: False
CIFAKE folder exists: False /content/ai-image-detector/data/raw/CIFAKE
SID_Set folder exists: False /content/ai-image-detector/data/raw/SID_Set_subset


## 2. Automated inspection

This section checks the files physically present under `data/raw/`.

In [ ]:
dimensions = Counter()
formats = Counter()
extensions = Counter()
images_per_dataset = Counter()
corrupt = []
duplicate_groups = defaultdict(list)


def file_hash(path):
    digest = hashlib.md5()

    with path.open("rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


for dataset_name, dataset_root in DATASET_ROOTS.items():

    if not dataset_root.exists():
        print(
            f"Skipping missing dataset: {dataset_root}"
        )
        continue

    for path in dataset_root.rglob("*"):

        if not path.is_file():
            continue

        if path.suffix.lower() not in VALID_EXTENSIONS:
            continue

        images_per_dataset[dataset_name] += 1
        extensions[path.suffix.lower()] += 1

        try:
            # Check whether the image is readable.
            with Image.open(path) as image:
                image.verify()

            # Reopen after verify().
            with Image.open(path) as image:
                dimensions[image.size] += 1
                formats[image.format] += 1

            duplicate_groups[
                file_hash(path)
            ].append(str(path.resolve()))

        except Exception as error:
            corrupt.append(
                (str(path), str(error))
            )


print("Images by dataset:")
for dataset_name, count in images_per_dataset.items():
    print(f"{dataset_name}: {count}")

print("\nTotal image files:")
print(sum(images_per_dataset.values()))

print("\nImage dimensions:")
for size, count in dimensions.most_common():
    print(f"{size}: {count}")

print("\nFile formats:")
for image_format, count in formats.most_common():
    print(f"{image_format}: {count}")

print("\nFile extensions:")
for extension, count in extensions.most_common():
    print(f"{extension}: {count}")

print("\nCorrupt images:", len(corrupt))

for path, error in corrupt[:20]:
    print(path, error)

Total image files: 120000

Image dimensions:
(32, 32): 120000

File formats:
JPEG: 120000

File extensions:
.jpg: 120000

Corrupt images: 0


## 3. Validate `sid_priority_splits.csv`

This checks how the repository intends to use the files: train, validation, and test, with labels REAL = 0 and FAKE = 1.

In [ ]:
split_counts = Counter()
class_counts = Counter()
source_counts = Counter()
image_records = []


def resolve_image_path(path_string):
    path = Path(str(path_string))

    if path.is_absolute():
        return path.resolve()

    return (REPO_ROOT / path).resolve()


with SPLITS_FILE.open(newline="") as file:
    rows = list(csv.DictReader(file))


required_columns = {
    "image_path",
    "label",
    "split",
}

if not rows:
    raise ValueError(
        f"No rows found in {SPLITS_FILE}"
    )

missing_columns = (
    required_columns
    - set(rows[0].keys())
)

print("CSV rows:", len(rows))
print("Missing required columns:", missing_columns)

for row in rows:

    image_path = resolve_image_path(
        row["image_path"]
    )

    label = int(row["label"])
    split = row["split"]

    class_name = row.get(
        "class_name",
        "REAL" if label == 0 else "FAKE",
    )

    source_dataset = row.get(
        "source_dataset",
        "unknown",
    )

    split_counts[split] += 1

    class_counts[
        (split, label, class_name)
    ] += 1

    source_counts[
        (source_dataset, split, label)
    ] += 1

    image_records.append({
        "path": str(image_path),
        "split": split,
        "label": label,
        "class_name": class_name,
        "source_dataset": source_dataset,
    })


print("\nSplit counts:")
for split, count in sorted(
    split_counts.items()
):
    print(f"{split}: {count}")

print("\nClass counts:")
for key, count in sorted(
    class_counts.items()
):
    print(f"{key}: {count}")

print("\nSource counts:")
for key, count in sorted(
    source_counts.items()
):
    print(f"{key}: {count}")

missing_paths = [
    record["path"]
    for record in image_records
    if not Path(record["path"]).exists()
]

print("\nMissing paths:", len(missing_paths))

for path in missing_paths[:20]:
    print(path)

## 4. Check duplicates and split leakage

Exact duplicate files are grouped by their MD5 hash. The critical problem is a duplicate appearing in different splits, such as train and test.

In [ ]:
duplicate_groups = {
    digest: paths
    for digest, paths in duplicate_groups.items()
    if len(paths) > 1
}

print(
    "Duplicate groups:",
    len(duplicate_groups),
)

for paths in list(
    duplicate_groups.values()
)[:10]:
    print(paths)


path_to_split = {
    record["path"]: record["split"]
    for record in image_records
}


cross_split_duplicates = []

for paths in duplicate_groups.values():

    splits = {
        path_to_split[path]
        for path in paths
        if path in path_to_split
    }

    if len(splits) > 1:
        cross_split_duplicates.append(
            (paths, splits)
        )


print(
    "\nDuplicate groups crossing splits:",
    len(cross_split_duplicates),
)

for paths, splits in cross_split_duplicates[:10]:
    print("Splits:", splits)
    print(paths)